# Clase 4 · 02 · Calidad y enriquecimiento sin estado
Esta consulta transforma cada entrada independientemente. Conserva tanto válidos como rechazados.
Los joins son stream-static con snapshots de clientes y productos.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala de clases 1 y 2")
dbutils.widgets.dropdown("expected_batch_id", "stream_001",
                         ["stream_001", "stream_002", "stream_003", "stream_004", "stream_005"], "03 - Etapa a validar")
dbutils.widgets.text("job_run_id", "interactive", "04 - ID de ejecución")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_stream_data

In [ ]:
%run ../common/streaming_support

In [ ]:
from pyspark.sql import functions as F
config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
repetitions = STREAM_REPETITIONS[config.scale]
expected_batch_id = normalize_stream_batch(dbutils.widgets.get("expected_batch_id"))
job_run_id = dbutils.widgets.get("job_run_id").strip()[:100] or "interactive"
spark.conf.set("spark.sql.session.timeZone", "UTC")
require_stream_lab(spark, config)

In [ ]:
%run ../common/streaming_rules

## 1. Contratos y joins
Revisá `common/streaming_rules.py`: from_json, try_cast, tipos y orden de motivos de rechazo.
El contrato acepta compras con importe positivo y controles de tiempo con importe cero.
Un evento con timestamp válido pero demasiado tardío **no es un error de calidad**.

In [ ]:
raw = spark.readStream.table(qualified_table(config, "bronze_stream_events"))
customers = spark.table(qualified_table(config, "stream_customers_snapshot"))
products = spark.table(qualified_table(config, "stream_products_snapshot"))
classified = classify_stream_events(raw, customers, products)
assert classified.isStreaming
classified.printSchema()

## 2. Persistir una única clasificación
Las vistas `silver_stream_valid` y `silver_stream_quarantine` filtran la misma tabla materializada.
Así no necesitamos coordinar dos escrituras dentro de foreachBatch para separar aceptados y rechazados.

In [ ]:
query = (classified.writeStream.format("delta").outputMode("append")
         .option("checkpointLocation", stream_checkpoint(config, "quality"))
         .queryName(f"{config.schema}_quality")
         .trigger(availableNow=True)
         .toTable(qualified_table(config, "silver_stream_classified")))
wait_and_record_stream(spark, config, query, "quality", job_run_id)

In [ ]:
%sql
SELECT source_batch_id, COALESCE(quality_reason, 'VALID') AS result, COUNT(*) AS rows
FROM silver_stream_classified GROUP BY source_batch_id, quality_reason ORDER BY source_batch_id, result

In [ ]:
%sql
SELECT event_id, quality_reason, raw_json FROM silver_stream_quarantine ORDER BY event_id LIMIT 10

### Para responder
¿Por qué se congelan las dimensiones? ¿Qué cambiaría si el país del cliente se modificara entre intentos?
¿Qué fila resulta válida por contrato pero puede descartarse luego por el watermark?